# Online Retail Sales Analysis and Interactive Dashboard

**Overview**

This project analyses a year of online retail transactions from a UK-based gift wholesaler. The raw data is cleaned, explored with a set of clear charts, and turned into an interactive dashboard a non-technical manager can use to explore revenue, products, countries and order patterns without touching any code.

**Business questions**

- How do sales change over time, and are there clear peak periods?
- Which products bring in the most revenue?
- Which countries drive the most sales?
- What does a typical order look like, and how large is the long tail of big orders?
- How do the number of orders and total customer spend relate to each other?

**Tools used**

Python, pandas, NumPy, Matplotlib, Plotly, ipywidgets and Voila.

**How to use the dashboard**

Scroll to the **Interactive Dashboard** section near the end. Use the three controls at the top of that section:

- **Period** to focus on a single month, or leave on *All*.
- **Country** to view one market, or leave on *All*.
- **Min Order** to hide very small orders.

The six charts update automatically whenever you change a control.


# Online Retail – Data Visualisation and Communication
This notebook explored the Online Retail dataset from a UK based gift wholesaler, covering invoice level transactions with product codes and descriptions, quantities, unit prices, timestamps, customer IDs and countries. The aim was to clean the raw data so it was reliable for analysis, use exploratory data analysis and static charts to spot sales patterns, and build an interactive dashboard that let a manager quickly explore revenue, products and countries. I explained the key steps and charts in plain language so a non technical manager could follow the logic and use the insights for decisions. 


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display

# set the matplotlib style to ggplot so the default charts look nice
plt.style.use("ggplot")
%matplotlib inline 

#online retail excel file
file_path = "Online Retail.xlsx"
df_raw = pd.read_excel(file_path)

# checked the first few rows to make sure the data loaded correctly
df_raw.head()


In [ ]:
# gives the shape of the data set meaning the number of rows and columns 
print("Shape:", df_raw.shape)
# output 
print("\nDtypes:\n", df_raw.dtypes)

# this code gives me summary stetistics for each column
df_raw.describe(include="all").T

# Analysis and understanding of the Data
Before I changed anything I checked the basics of the Online Retail data, how many rows and columns there are, the data type for each field, and quick summaries for quantity and price. The file has hundreds of thousands of invoice lines with fields like InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID and Country. Straight away I saw issues that could mess up charts later on like missing values especially in CustomerID and Description, negative quantities or odd unit prices that usually mean returns or errors, and duplicate lines where the same invoice and product show up twice. Writing this first matters because insights like top products or revenue by country only make sense if we clean the data properly.

In [ ]:
# this checks for missing values in e column
df_raw.isnull().sum()


In [ ]:
#   this checks for duplicates
df_raw.duplicated().sum()


In [ ]:
# looked at negative or zero quantities and  prices to see errors as they wont be sales probablly or maybe returns
df_raw[(df_raw["Quantity"] <= 0) | (df_raw["UnitPrice"] <= 0)].head()


# 1-Data Cleaning and Prep
Here i started by making a working copy so the original stayed untouched. Then I converted InvoiceDate to a proper datetime so I could group by date and build time series later. I removed exact duplicates to avoid double counting and filtered out rows where Quantity or UnitPrice were non positive since these usually meant cancelled orders returns or entry mistakes. That kept the focus on positive completed sales so figures were gross and dont subtract returns. I used a simple IQR rule to flag unusually low or high values in Quantity and UnitPrice and at that point I just counted them. For some plots I capped values at the 99th percentile so a few extreme orders didnt squash everything else. I created LineTotal = Quantity × UnitPrice for line revenue. For customer analysis I dropped rows with missing CustomerID and I wrote down what I removed and why so the cleaning stayed transparent


In [ ]:
# made a copy of the raw data so i could clean it without touching the original dataframe
df = df_raw.copy()

# Convert InvoiceDate to datetime
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Drop exact duplicate rows
print("Duplicates before:", df.duplicated().sum())
df = df.drop_duplicates()
# checked again to confirm that all duplicates had been removed
print("Duplicates after:", df.duplicated().sum())


In [ ]:
# this counts invalid rows
# condition is if the row is less then or equal to 0 then return true otherwise false. This way i will get only the invalid rows  
invalid_mask = (df["Quantity"] <= 0) | (df["UnitPrice"] <= 0)
print("Invalid rows (Quantity<=0 or UnitPrice<=0):", invalid_mask.sum())

# this will remove them letting me focus on positive sales lines
df = df[~invalid_mask]

df.shape


In [ ]:
# # this finds out a normal range for quantity and unitprice and then shows how many values are unusually low or high which can be possible outliers


def iqr_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return lower, upper

for col in ["Quantity", "UnitPrice"]:
    lower, upper = iqr_bounds(df[col])
    print(f"{col}: lower={lower:.2f}, upper={upper:.2f}")
    print("Below lower:", (df[col] < lower).sum(),
          "| Above upper:", (df[col] > upper).sum())


In [ ]:
# added a new column with the total value of each transaction line by multiplying quantity by unit price
df["LineTotal"] = df["Quantity"] * df["UnitPrice"]
# checked the first few rows of quantity, unit price and line total to make sure the calculation looked correct
df[["Quantity", "UnitPrice", "LineTotal"]].head()


# Data Cleaned
After cleaning, duplicates are gone, rows with non positive Quantity or UnitPrice are dropped, InvoiceDate is stored as a proper datetime, and LineTotal adds line level revenue. I used the IQR rule to check for outliers in Quantity and UnitPrice. Whether to trim extremes depends on the business view any trimming is clearly recorded in the notebook.



In [ ]:
# Here I am checking how many unique invoices,products,customers,countries I have after cleaning, and then 
#I look at the basic stats for Quantity, UnitPrice and LineTotal so I can see the typical ranges and check if the numbers look alright.

print("Unique invoices:", df["InvoiceNo"].nunique())
print("Unique products:", df["StockCode"].nunique())
print("Unique customers:", df["CustomerID"].nunique())
print("Countries:", df["Country"].nunique())

df[["Quantity", "UnitPrice", "LineTotal"]].describe()


# 2-EDA 
# Daily revenue over time
I summed LineTotal by InvoiceDate to get daily revenue then plotted a line chart to show how sales change over time. A line chart fits because the x axis is time and we care about trends not single points. People read lines as continuous trends and use the slope to see rises and falls (Zacks & Tversky, 1999; Reimers & Harvey, 2023). The plot shows day to day swings with a few big spikes likely busy trading days or bulk orders. For a manager this helps answer if sales are generally stable and whether there are clear peak periods


In [ ]:
#checked if a pure date column already existed and if not i created InvoiceDateDate from the invoice datetime
if "InvoiceDateDate" not in df.columns:
    df["InvoiceDateDate"] = df["InvoiceDate"].dt.date

# grouped by date and summing line totals to get daily revenue
daily_revenue = (
    df.groupby("InvoiceDateDate")["LineTotal"]
      .sum()
      .reset_index()
)

# setted up the figure and axes for the daily revenue line chart
fig, ax = plt.subplots(figsize=(20, 6))

# plotted daily revenue over time
ax.plot(daily_revenue["InvoiceDateDate"],
        daily_revenue["LineTotal"],
        color="teal")  

ax.set_title("Daily Revenue Over Time (Gross Sales)", fontsize=12)
ax.set_xlabel("Date")
ax.set_ylabel("Revenue")
# left the x axis labels horizontal
ax.tick_params(axis="x",)

# added a light grid
ax.grid(alpha= 0.5)

plt.tight_layout()
plt.show()


# Top 10 products by revenue
I grouped the cleaned data by product description and summed LineTotal for each item then took the top ten by revenue and plotted a horizontal bar chart so a manager can quickly see which items make the most money. A bar chart fits because the job is to compare totals across a small set of categories and people judge lengths on a common baseline more accurately than angles or areas (Cleveland & McGill, 1984; Talbot et al., 2014). Recent work also backs bars for this kind of comparison and decision making (Franconeri et al., 2021; Xiong et al., 2022). The chart shows a few products drive a big share of revenue which helps focus stock promos and website placement and flags items that add little



In [ ]:
# calculted the top 10 products by total revenue which shows which items bring in the most money
product_revenue = (
    df.groupby(["StockCode", "Description"], dropna=False)["LineTotal"]
      .sum()
      .reset_index()
      .sort_values("LineTotal", ascending=False)
      .head(10)
)

# setted up the figure and axes for a horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 5))

# ploted a horizontal bar chart of the top 10 products by revenue
ax.barh(
    product_revenue["Description"],
    product_revenue["LineTotal"],
    color="purple" 
)

ax.set_title("Top 10 Products by Revenue (€)", fontsize=12)
ax.set_xlabel("Revenue (€)")
ax.set_ylabel("Product")

# i inverted the y axis so the product with the highest revenue shows at the top of the chart
ax.invert_yaxis()

#made the product labels a bit smaller so long names fit on the plot
ax.tick_params(axis="y", labelsize=8)

# here I added the exact revenue numbers at the end of each bar so itss easier to read the values
for i, v in enumerate(product_revenue["LineTotal"]):
    ax.text(v, i, f"{v:,.0f}", va="center", ha="left", fontsize=8)

plt.tight_layout()
plt.show()


# Order values – histogram
To see typical order size I summed LineTotal for each InvoiceNo to get order totals and drew a histogram, capping very large values at the 99th percentile so the plot stays readable. A histogram groups values into bins so you can see how often different order sizes happen its a standard way to show a distribution (Li et al., 2020) and to spot skew and long tails (Sahann et al., 2021; Silveira & Siqueira, 2022). The chart shows most orders are low value with a long tail of big ones, so many small purchases drive the business and fewer large baskets do the rest which helps when planning segments promos or minimum order thresholds



In [ ]:
#  orders table that summarises each invoice into one row
if "orders" not in globals():
    orders = (
        df.groupby("InvoiceNo")
          .agg(
              OrderDate=("InvoiceDate", "min"),
              Country=("Country", "first"),
              CustomerID=("CustomerID", "first"),
              OrderValue=("LineTotal", "sum"),
              NumLines=("StockCode", "nunique")
          )
          .reset_index()
    )
    # added a date only version of the order date so i can use it later if needed
    orders["OrderDateDate"] = orders["OrderDate"].dt.date

# this capps order values at the 99th percentile so extreme outliers dont squash the histogram
cap = orders["OrderValue"].quantile(0.99)

# setting up the figure and axes for the histogram of order values
fig, ax = plt.subplots(figsize=(12, 4))

# plotted the distribution of order values 
ax.hist(
    orders.loc[orders["OrderValue"] <= cap, "OrderValue"],
    bins=30,
    edgecolor="black",
    alpha=0.85,
    color="cyan"  
)

ax.set_title("Distribution of Order Values (≤ 99th percentile)", fontsize=12)
ax.set_xlabel("Order value (€)")
ax.set_ylabel("Number of orders")

# added a light grid
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


# Revenue by country
To see which markets matter most I grouped the data by Country summed LineTotal and plotted a bar chart of the top countries. A bar chart fits because we mainly want to compare revenue across categories and people judge lengths from a shared baseline quickly and accurately (Cleveland & McGill, 1984; Talbot et al., 2014). Reviews also recommend bars as a sensible default for category comparisons for non experts (Franconeri et al., 2021; Xiong et al., 2022). The chart shows the United Kingdom dominates revenue while other countries contribute much less, which is a strength for the home market but also a risk if we rely too much on one country


In [ ]:
# calculated total revenue per country and sorted it so i could see which countries spent the most
country_revenue = (
    df.groupby("Country")["LineTotal"]
      .sum()
      .reset_index()
      .sort_values("LineTotal", ascending=False)
)

# took the top 10 countries by revenue for bar chart
top_countries = country_revenue.head(10)

# setted up the figure and axes for the bar chart of top countries
fig, ax = plt.subplots(figsize=(12, 6))

# plotted a bar chart showing the top 10 countries by revenue
ax.bar(
    top_countries["Country"],
    top_countries["LineTotal"],
    color="pink"  
)

ax.set_title("Top 10 Countries by Revenue (€)", fontsize=12)
ax.set_xlabel("Country")
ax.set_ylabel("Revenue (€)")

# left the x axis labels horizontal so the country names were easy to read
ax.tick_params(axis="x")

# added the exact revenue values above each bar so the amounts are easy to see
for x, y in zip(top_countries["Country"], top_countries["LineTotal"]):
    ax.text(x, y, f"{y:,.0f}", ha="center", va="bottom", fontsize=8)

# added a horizontal grid to help compare bar heights
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


# Heat Map
I built a correlation matrix for OrderValue, NumLines and TotalQuantity and plotted it as a heatmap. Each cell shows how strong and in what direction the linear link is between two metrics. A heatmap works here because it squeezes the whole matrix into a small grid and uses colour so non technical readers can spot strong or weak links fast. The literature also recommends heatmaps for matrix shaped data (Franconeri et al., 2021). The pattern looks sensible, order value is strongly and positively related to total quantity and number of lines, while some other pairs are only weakly linked. This quick check reassures us the metrics behave as expected and backs how we read the rest of the dashboard


In [ ]:
# selected the numeric columns that i wanted to look and show the relationships for
numeric_cols = ["Quantity", "UnitPrice", "LineTotal"]

# calculated the correlation matrix between quantity, unit price and line total
corr = df[numeric_cols].corr()

# set up the figure and axes for a small heatmap plot
fig, ax = plt.subplots(figsize=(10, 7))

# plotted the correlation matrix as a heatmap using a coolwarm colour map so I can see positive vs negative relationships
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)

# set the x tick positions and labels to match the numeric columns
ax.set_xticks(np.arange(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, ha="right")

# set the y tick positions and labels to match the numeric columns
ax.set_yticks(np.arange(len(numeric_cols)))
ax.set_yticklabels(numeric_cols)

# added the actual correlation number inside each heatmap cell
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(
            j, i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center", va="center",
            fontsize=9,
            color="black"
        )

# added a colour bar on the side to show the scale of correlation values from -1 to 1
fig.colorbar(im, ax=ax, label="Correlation")

# seted the title
ax.set_title("Correlation Heatmap (Quantity, UnitPrice, LineTotal)", fontsize=12)

plt.tight_layout()
plt.show()


# 3-Interactive Dashboard 
This section builds a simple interactive sales dashboard with ipywidgets and matplotlib so its possiable to tweak filters and see the charts update, which suits a non technical sales or ops manager who wants what if answers without writing code. I first rolled the transactions into an orders table where each row is one invoice with date, country, customer ID, total quantity, number of product lines and total order value. The dashboard shows a 2 × 3 grid of charts covering daily revenue over time, top products by revenue, revenue by country, the distribution of order values, a customer orders versus total spend scatter, and a correlation heatmap for key numeric fields. Filters at the top let us choose a period, pick countries like just the UK or all and set a minimum order value to ignore tiny baskets. A small helper applies those filters to the orders table before every plot so everything stays in sync. Limits to keep in mind it uses historical transactions only, not marketing, stock levels or post invoice returns, and a few very large orders can still pull the view. Its descriptive not predictive, meant to support quick understanding and decisions rather than forecasting.


In [ ]:
# 3 interactive Dashboard

# grouped the cleaned line level data by invoice number to build an order level table for the dashboard
orders = (
    df.groupby("InvoiceNo")
      .agg(
          OrderDate=("InvoiceDate", "min"),
          Country=("Country", "first"),
          CustomerID=("CustomerID", "first"),
          OrderValue=("LineTotal", "sum"),
          NumLines=("StockCode", "nunique"),
          TotalQuantity=("Quantity", "sum")
      )
      .reset_index()
)

# added a pure date version of the order date so i could use it easily in daily plots
orders["OrderDateDate"] = orders["OrderDate"].dt.date
# created a YearMonth string column so i could filter the dashboard by month and year
orders["YearMonth"] = orders["OrderDate"].dt.to_period("M").astype(str)

orders.head()


In [ ]:
# dropdown for year-month period with options starting with All so its possiable to see everything or pick a specific period
yearmonth_options = ["All"] + sorted(orders["YearMonth"].unique().tolist())
yearmonth_dropdown = widgets.Dropdown(
    options=yearmonth_options,
    value="All",
    description="Period:",
    layout=widgets.Layout(width="200px")
)

# dropdown for country that starts with All and then listed each country in alphabetical order
country_options = ["All"] + sorted(orders["Country"].dropna().unique().tolist())
country_dropdown = widgets.Dropdown(
    options=country_options,
    value="All",
    description="Country:",
    layout=widgets.Layout(width="250px")
)

# slider for minimum order value to control and easily filter out very small orders from the dashboard
min_order_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=int(orders["OrderValue"].quantile(0.95)),
    step=10,
    description="Min Order:",
    continuous_update=False,
    layout=widgets.Layout(width="300px")
)



In [ ]:
# colours for graphs
color_line_revenue = "#1f77b4"         # blue
color_bar_products = "#ff7f0e"      # orange
color_bar_countries = "#2ca02c"      # green
color_hist_orders = "#9467bd"     # purple
color_scatter_customers = "#d62728"  # red


# filtering function to filter the orders table based on the dashboard controls

def filter_data(period, country, min_order):
    """
    Filter the orders table based on dashboard controls.
    period: YearMonth string or "All"
    country: country name or "All"
    min_order: minimum OrderValue
    """
    filtered = orders.copy()

# if a specific period was picked this keeps only those rows
    if period != "All":
        filtered = filtered[filtered["YearMonth"] == period]
# if a specific country was picked this keeps only orders from that country
    if country != "All":
        filtered = filtered[filtered["Country"] == country]
#  this will filtere out orders below the chosen minimum order value
    filtered = filtered[filtered["OrderValue"] >= min_order]

    return filtered



#  dashboard update function that rebuilts all charts when filters changed

def update_dashboard(period, country, min_order):
# filtered order level data using the current widget values
    data_orders = filter_data(period, country, min_order)

# if there were no orders after filtering this will show a simple message instead of empty charts
    if data_orders.empty:
        print("No orders match your filters.")
        return

    #  pulls the line level rows for the invoices that are still in the filtered orders
    data_lines = df[df["InvoiceNo"].isin(data_orders["InvoiceNo"])]

    # createed 2 x 3 grid of axes 
    fig, axes = plt.subplots(2, 3, figsize=(28, 12))
    ax1, ax2, ax3 = axes[0]
    ax4, ax5, ax6 = axes[1]

    # [1] Revenue over time (daily line chart)
    daily = (
        data_orders.groupby("OrderDateDate")["OrderValue"]
                   .sum()
                   .reset_index()
                   .sort_values("OrderDateDate")
    )
    ax1.plot(daily["OrderDateDate"], daily["OrderValue"], color=color_line_revenue)
    ax1.set_title("Revenue Over Time", fontsize=11)
    ax1.set_xlabel("Date")
    ax1.set_ylabel("Revenue (€)")
    ax1.tick_params(axis="x", rotation=45)
    ax1.grid(alpha=0.3)

    # [2] Top products by revenue (barh)
    prod_rev = (
        data_lines.groupby(["StockCode", "Description"], dropna=False)["LineTotal"]
                  .sum()
                  .reset_index()
                  .sort_values("LineTotal", ascending=False)
                  .head(10)
    )
    ax2.barh(
        prod_rev["Description"],
        prod_rev["LineTotal"],
        color=color_bar_products
    )
    ax2.set_title("Top 10 Products by Revenue", fontsize=11)
    ax2.set_xlabel("Revenue (€)")
    ax2.set_ylabel("Product")
# inverted the y axis so the highest revenue product appeares at the top
    ax2.invert_yaxis()
    ax2.tick_params(axis="y", labelsize=8)

    # [3] Revenue by country (bar)
    country_rev = (
        data_orders.groupby("Country")["OrderValue"]
                   .sum()
                   .reset_index()
                   .sort_values("OrderValue", ascending=False)
    )
    ax3.bar(
        country_rev["Country"],
        country_rev["OrderValue"],
        color=color_bar_countries
    )
    ax3.set_title("Revenue by Country", fontsize=11)
    ax3.set_xlabel("Country")
    ax3.set_ylabel("Revenue (€)")
    ax3.tick_params(axis="x", rotation=45)
 # right aligned the x labels so they were easier to read when rotated
    for label in ax3.get_xticklabels():
        label.set_ha("right")
 # added a light horizontal grid 
    ax3.grid(axis="y", alpha=0.3)

    # [4] Order value distribution (histogram)
    ax4.hist(
        data_orders["OrderValue"],
        bins=20,
        edgecolor="black",
        alpha=0.85,
        color=color_hist_orders
    )
    ax4.set_title("Order Value Distribution", fontsize=11)
    ax4.set_xlabel("Order value (€)")
    ax4.set_ylabel("Number of orders")
    ax4.grid(alpha=0.3)

    # [5] Customer orders vs total spend (scatter)
    cust_stats = (
        data_orders.groupby("CustomerID")
                   .agg(
                       TotalSpend=("OrderValue", "sum"),
                       NumOrders=("InvoiceNo", "nunique")
                   )
                   .reset_index()
    )

    # cap very large spends for readability
    spend_cap = cust_stats["TotalSpend"].quantile(0.99)
    mask = cust_stats["TotalSpend"] <= spend_cap

    ax5.scatter(
        cust_stats.loc[mask, "NumOrders"],
        cust_stats.loc[mask, "TotalSpend"],
        alpha=0.6,
        color=color_scatter_customers
    )
    ax5.set_title("Customer Orders vs Total Spend", fontsize=11)
    ax5.set_xlabel("Number of Orders")
    ax5.set_ylabel("Total Spend (€)")
    ax5.grid(alpha=0.3)

    # [6] Correlation heatmap (OrderValue, NumLines, TotalQuantity)
    numeric_cols = ["OrderValue", "NumLines", "TotalQuantity"]
    corr = data_orders[numeric_cols].corr()

    im = ax6.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
    ax6.set_title("Correlation Heatmap", fontsize=11)
    ax6.set_xticks(range(len(numeric_cols)))
    ax6.set_yticks(range(len(numeric_cols)))
    ax6.set_xticklabels(numeric_cols, rotation=0)
    ax6.set_yticklabels(numeric_cols)

    # added the actual correlation values into each cell so I could read the numbers as well as see the colours
    for i in range(len(numeric_cols)):
        for j in range(len(numeric_cols)):
            ax6.text(
                j, i,
                f"{corr.iloc[i, j]:.2f}",
                ha="center", va="center", color="black", fontsize=9
            )
# added a colour bar to show the scale of the correlations from -1 to 1
    fig.colorbar(im, ax=ax6, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()


In [ ]:
# created a horizontal box layout so the year - month, country and minimum order controls sat togather
controls = widgets.HBox([
    yearmonth_dropdown,
    country_dropdown,
    min_order_slider
])

# wired the interactive widgets up to the update_dashboard function so changing any control refreshed the charts
out = widgets.interactive_output(
    update_dashboard,
    {
        "period": yearmonth_dropdown,
        "country": country_dropdown,
        "min_order": min_order_slider
    }
)

# displayed the controls row on top and the live dashboard output
display(controls, out)


# Refrences
Cleveland, W.S. and McGill, R. (1984) ‘Graphical perception: theory, experimentation and application to the development of graphical methods’, Journal of the American Statistical Association, 79(387), pp. 531–554. Available at: https://doi.org/10.1080/01621459.1984.10478080 

Franconeri, S.L., Padilla, L.M., Shah, P., Zacks, J.M. and Hullman, J. (2021) ‘The science of visual data communication: what works’, Psychological Science in the Public Interest, 22(3), pp. 110–161. Available at: https://doi.org/10.1177/15291006211051956686 

Li, H., Munk, A., Sieling, H. and Walther, G. (2020) ‘The essential histogram’, Biometrika, 107(2), pp. 347–364. Available at: https://academic.oup.com/biomet/article-abstract/107/2/347/5733266 

Reimers, S. and Harvey, N. (2023) ‘Bars, lines and points: the effect of graph format on judgmental forecasting’, International Journal of Forecasting, 40(1), pp. 44–61. Available at: https://www.sciencedirect.com/science/article/pii/S0169207022001467 

Sahann, R., Möller, T. and Schmidt, J. (2021) ‘Histogram binning revisited with a focus on human perception’, in IEEE VIS Short Papers 2021. Available at: https://arxiv.org/abs/2109.06612 

Silveira, P.S.P. and Siqueira, J.O. (2022) ‘Histogram lies about distribution shape and Pearson’s coefficient of variation lies about relative variability’, The Quantitative Methods for Psychology, 18(1), pp. 91–111. Available at: https://doi.org/10.20982/tqmp.18.1.p091 

Talbot, J., Setlur, V. and Anand, A. (2014) ‘Four experiments on the perception of bar charts’, IEEE Transactions on Visualization and Computer Graphics, 20(12), pp. 2152–2160. Available at: https://ieeexplore.ieee.org/document/6876021 

Xiong, C., Lee-Robbins, E., Zhang, I., Gaba, A. and Franconeri, S. (2022) ‘Reasoning affordances with tables and bar charts’, IEEE Transactions on Visualization and Computer Graphics (in press). Available at: https://ieeexplore.ieee.org/d

Project Jupyter (2023) Jupyter Widgets (ipywidgets) – documentation. Available at: https://ipywidgets.readthedocs.io/en/latest/ 

Stack Overflow (2016) ‘Using IPython ipywidget to create a variable?’, Stack Overflow. Available at: https://stackoverflow.com/questions/35361038/using-ipython-ipywidget-to-create-a-variable.3758/BF03201236 

